In [16]:
# 203. Sort the feature data by time and perform a chronological train/test split: earlier periods train, later periods test. Never a random split.

import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="nopis"
)

query = """
SELECT
    grid_id,
    feature_timestamp,
    avg_activity,
    activity_growth,
    active_hours,
    peak_ratio,
    variability,
    internet_share_avg,
    next_total_activity
FROM network_feature_table
ORDER BY feature_timestamp, grid_id
"""

ml_df = pd.read_sql_query(query, conn)

conn.close()

ml_df["feature_timestamp"] = pd.to_datetime(
    ml_df["feature_timestamp"],
    format="%Y%m%d%H"
)

print("Rows loaded:", len(ml_df))
print("First timestamp:", ml_df["feature_timestamp"].min())
print("Last timestamp:", ml_df["feature_timestamp"].max())

display(ml_df.head())

D:\NOPIS\tmp\ipykernel_5700\1990750058.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ml_df = pd.read_sql_query(query, conn)


Rows loaded: 1559994
First timestamp: 2013-11-01 11:00:00
Last timestamp: 2013-11-07 22:00:00


,grid_id,feature_timestamp,avg_activity,activity_growth,active_hours,peak_ratio,variability,internet_share_avg,next_total_activity
0,1,2013-11-01 11:00:00,71.331967,29.123717,6.0,1.412602,26.633648,0.883292,91.2744
1,2,2013-11-01 11:00:00,71.757750,29.447717,6.0,1.414766,26.890547,0.882464,91.8687
2,3,2013-11-01 11:00:00,72.210867,29.792567,6.0,1.417040,27.164007,0.881595,92.5010
3,4,2013-11-01 11:00:00,70.099133,28.185633,6.0,1.406191,25.889938,0.885746,89.5540
4,5,2013-11-01 11:00:00,63.956850,25.770550,6.0,1.406284,23.668961,0.882827,82.1570


In [3]:
# 203. Sort the feature data by time and perform a chronological train/test split: earlier periods train, later periods test. Never a random split.

# Sort by timestamp first
ml_df = ml_df.sort_values(
    ["feature_timestamp", "grid_id"]
).reset_index(drop=True)

# Get unique dates
unique_dates = sorted(
    ml_df["feature_timestamp"].dt.date.unique()
)

# First 6 days = training, remaining final day = testing
train_dates = unique_dates[:6]
test_dates = unique_dates[-1:]

train_df = ml_df[
    ml_df["feature_timestamp"].dt.date.isin(train_dates)
].copy()

test_df = ml_df[
    ml_df["feature_timestamp"].dt.date.isin(test_dates)
].copy()

print("Training dates:", train_dates)
print("Testing date:", test_dates)

print("\nTrain rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain period:")
print(train_df["feature_timestamp"].min(), "to", train_df["feature_timestamp"].max())

print("\nTest period:")
print(test_df["feature_timestamp"].min(), "to", test_df["feature_timestamp"].max())

Training dates: [datetime.date(2013, 11, 1), datetime.date(2013, 11, 2), datetime.date(2013, 11, 3), datetime.date(2013, 11, 4), datetime.date(2013, 11, 5), datetime.date(2013, 11, 6)]
Testing date: [datetime.date(2013, 11, 7)]

Train rows: 1329995
Test rows: 229999

Train period:
2013-11-01 11:00:00 to 2013-11-06 23:00:00

Test period:
2013-11-07 00:00:00 to 2013-11-07 22:00:00


In [4]:
# 204. Prepare the dataset by removing rows without complete features or a t+1 target.

feature_columns = [
    "avg_activity",
    "activity_growth",
    "active_hours",
    "peak_ratio",
    "variability",
    "internet_share_avg"
]

target_column = "next_total_activity"

# Remove rows with missing features or missing t+1 target
train_df = train_df.dropna(
    subset=feature_columns + [target_column]
).copy()

test_df = test_df.dropna(
    subset=feature_columns + [target_column]
).copy()

print("Train rows after cleaning:", len(train_df))
print("Test rows after cleaning:", len(test_df))

print("\nMissing values in train:")
print(train_df[feature_columns + [target_column]].isna().sum())

print("\nMissing values in test:")
print(test_df[feature_columns + [target_column]].isna().sum())

Train rows after cleaning: 1329995
Test rows after cleaning: 229999

Missing values in train:
avg_activity           0
activity_growth        0
active_hours           0
peak_ratio             0
variability            0
internet_share_avg     0
next_total_activity    0
dtype: int64

Missing values in test:
avg_activity           0
activity_growth        0
active_hours           0
peak_ratio             0
variability            0
internet_share_avg     0
next_total_activity    0
dtype: int64


In [5]:
# 204. Record and report the earliest and latest timestamp in each of the train and test sets.

print("TRAIN SET")
print("Earliest:", train_df["feature_timestamp"].min())
print("Latest:  ", train_df["feature_timestamp"].max())

print("\nTEST SET")
print("Earliest:", test_df["feature_timestamp"].min())
print("Latest:  ", test_df["feature_timestamp"].max())

TRAIN SET
Earliest: 2013-11-01 11:00:00
Latest:   2013-11-06 23:00:00

TEST SET
Earliest: 2013-11-07 00:00:00
Latest:   2013-11-07 22:00:00


In [6]:
# 204. Verify that train and test time periods do not overlap.

assert train_df["feature_timestamp"].max() < test_df["feature_timestamp"].min()

print("PASS: Train and test periods do not overlap.")

PASS: Train and test periods do not overlap.


In [7]:
#Calculate the target threshold using only the training period.

TARGET_THRESHOLD = train_df["next_total_activity"].quantile(0.95)

print("Training-period 95th percentile threshold:", TARGET_THRESHOLD)

# Create binary target using the training-derived threshold
train_df["target"] = (
    train_df["next_total_activity"] > TARGET_THRESHOLD
).astype(int)

test_df["target"] = (
    test_df["next_total_activity"] > TARGET_THRESHOLD
).astype(int)

print("\nTrain target distribution:")
print(train_df["target"].value_counts())

print("\nTest target distribution:")
print(test_df["target"].value_counts())

print("\nTrain target percentage:")
print((train_df["target"].value_counts(normalize=True) * 100).round(2))

print("\nTest target percentage:")
print((test_df["target"].value_counts(normalize=True) * 100).round(2))

Training-period 95th percentile threshold: 1893.4091300000007

Train target distribution:
target
0    1263495
1      66500
Name: count, dtype: int64

Test target distribution:
target
0    215599
1     14400
Name: count, dtype: int64

Train target percentage:
target
0    95.0
1     5.0
Name: proportion, dtype: float64

Test target percentage:
target
0    93.74
1     6.26
Name: proportion, dtype: float64


In [8]:
# Prepare training and test feature matrices and target vectors.

X_train = train_df[feature_columns]
y_train = train_df["target"]

X_test = test_df[feature_columns]
y_test = test_df["target"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nFeatures used:")
print(feature_columns)

X_train shape: (1329995, 6)
y_train shape: (1329995,)
X_test shape: (229999, 6)
y_test shape: (229999,)

Features used:
['avg_activity', 'activity_growth', 'active_hours', 'peak_ratio', 'variability', 'internet_share_avg']


In [10]:
# 205. Train Logistic Regression or a Decision Tree.
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [11]:
# 206. Evaluate accuracy plus precision, recall and the class balance. Report the base rate — the proportion of positive labels — alongside accuracy, because accuracy is meaningless without it.

from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# Make predictions on the unseen test data
y_pred = model.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

# Class balance / base rate
positive_count = y_test.sum()
negative_count = (y_test == 0).sum()
base_rate = y_test.mean()

print("TEST SET EVALUATION")
print("-------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")

print("\nCLASS BALANCE")
print("-------------")
print(f"Negative cases: {negative_count:,}")
print(f"Positive cases: {positive_count:,}")
print(f"Base rate     : {base_rate:.4f} ({base_rate * 100:.2f}%)")

TEST SET EVALUATION
-------------------
Accuracy : 0.9767
Precision: 0.8755
Recall   : 0.7322

CLASS BALANCE
-------------
Negative cases: 215,599
Positive cases: 14,400
Base rate     : 0.0626 (6.26%)


Precision 87.84% → when the model flags a risk, about 88% of those flags are actually positive according to our proxy label.  
Recall 80.67% → the model catches about 81% of the positive cases.  
Base rate 12.05% → only about 12% of test cases are positive.

In [12]:
# 206. Inspect the confusion matrix to investigate the high accuracy.

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[214100   1499]
 [  3857  10543]]


                 Predicted
                 0       1
Actual 0            182473   2836  
Actual 1              4908  20484

In [13]:
# 207. Inspect the coefficients or the feature importances and check they are operationally plausible.

coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": model.coef_[0]
})

coefficients["direction"] = coefficients["coefficient"].apply(
    lambda x: "Positive risk" if x > 0 else "Negative risk"
)

coefficients.sort_values(
    "coefficient",
    ascending=False
)

,feature,coefficient,direction
5,internet_share,5.125015,Positive risk
3,peak_ratio,2.836727,Positive risk
0,avg_activity,0.003714,Positive risk
1,activity_growth,0.001126,Positive risk
4,variability,-0.000943,Negative risk
2,active_hours,-2.493692,Negative risk


In [14]:
# 207. Display the Logistic Regression coefficients.

print(coefficients.to_string(index=False))

        feature  coefficient     direction
   avg_activity     0.003714 Positive risk
activity_growth     0.001126 Positive risk
   active_hours    -2.493692 Negative risk
     peak_ratio     2.836727 Positive risk
    variability    -0.000943 Negative risk
 internet_share     5.125015 Positive risk


avg_activity +0.0081 → higher recent activity increases predicted risk. Plausible.

activity_growth +0.5209 → increasing activity increases predicted risk. Plausible.

active_hours −0.0278 → more consistently active hours slightly reduce predicted risk. Less intuitive, but possible given the other features.

peak_ratio +2.0627 → stronger peaks increase predicted risk. Plausible.

variability −0.0068 → higher variation slightly reduces predicted risk. Not obviously expected, so this should be noted rather than over-interpreted.

internet_share −11.3481 → higher internet share strongly reduces predicted risk. Needs caution, because this is a strong relationship learned from this dataset and proxy label, not proof of a real network relationship.

In [20]:
# 208. Load the raw network activity data required for comparison with NP3.

import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="nopis"
)

query = """
SELECT
    grid_id,
    time_key AS activity_timestamp,
    total_activity
FROM fact_network_activity
ORDER BY grid_id, time_key
"""

activity_df = pd.read_sql_query(query, conn)

conn.close()

activity_df["activity_timestamp"] = pd.to_datetime(
    activity_df["activity_timestamp"],
    format="%Y%m%d%H"
)

print("Activity rows:", len(activity_df))
print("First timestamp:", activity_df["activity_timestamp"].min())
print("Last timestamp:", activity_df["activity_timestamp"].max())

display(activity_df.head())

D:\NOPIS\tmp\ipykernel_5700\3758781673.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  activity_df = pd.read_sql_query(query, conn)


Activity rows: 1679994
First timestamp: 2013-11-01 00:00:00
Last timestamp: 2013-11-07 23:00:00


,grid_id,activity_timestamp,total_activity
0,1,2013-11-01 00:00:00,62.0092
1,1,2013-11-01 01:00:00,46.3654
2,1,2013-11-01 02:00:00,42.0870
3,1,2013-11-01 03:00:00,35.0978
4,1,2013-11-01 04:00:00,32.2741


In [21]:
# 208. Compare the predictions against the rule-based NP3 alerts and characterize where they disagree.

# Create test comparison data
comparison_df = test_df[
    ["grid_id", "feature_timestamp"]
].copy()

comparison_df["ml_prediction"] = y_pred

# Prepare actual next-hour activity
next_activity = activity_df[
    ["grid_id", "activity_timestamp", "total_activity"]
].copy()

next_activity["feature_timestamp"] = (
    next_activity["activity_timestamp"] - pd.Timedelta(hours=1)
)

# Merge t+1 activity onto the test rows
comparison_df = comparison_df.merge(
    next_activity[
        ["grid_id", "feature_timestamp", "total_activity"]
    ],
    on=["grid_id", "feature_timestamp"],
    how="left"
)

# Create date for NP3-style baseline
next_activity["date"] = next_activity["activity_timestamp"].dt.date

# Calculate daily median baseline
next_activity["baseline_activity"] = (
    next_activity
    .groupby(["grid_id", "date"])["total_activity"]
    .transform("median")
)

# Prepare baseline timestamp for comparison
next_activity["feature_timestamp"] = (
    next_activity["activity_timestamp"] - pd.Timedelta(hours=1)
)

# Merge baseline
comparison_df = comparison_df.drop(
    columns=["baseline_activity"],
    errors="ignore"
)

comparison_df = comparison_df.merge(
    next_activity[
        ["grid_id", "feature_timestamp", "baseline_activity"]
    ],
    on=["grid_id", "feature_timestamp"],
    how="left"
)

# NP3 HIGH_ACTIVITY rule
comparison_df["np3_alert"] = (
    comparison_df["total_activity"]
    >= comparison_df["baseline_activity"] * 1.50
).astype(int)

print(comparison_df.head())

print("\nMissing next-hour activity:",
      comparison_df["total_activity"].isna().sum())

print("Missing NP3 baseline:",
      comparison_df["baseline_activity"].isna().sum())

   grid_id feature_timestamp  ml_prediction  total_activity  \
0        1        2013-11-07              0         44.6076   
1        2        2013-11-07              0         44.6970   
2        3        2013-11-07              0         44.7920   
3        4        2013-11-07              0         44.3489   
4        5        2013-11-07              0         40.4899   

   baseline_activity  np3_alert  
0            64.9008          0  
1            65.4995          0  
2            66.1367          0  
3            63.1671          0  
4            58.6355          0  

Missing next-hour activity: 1
Missing NP3 baseline: 1


In [22]:
# 208. Compare the predictions against the rule-based NP3 alerts and characterize where they disagree.

# Remove rows where t+1 activity or NP3 baseline is unavailable
comparison_df = comparison_df.dropna(
    subset=["total_activity", "baseline_activity"]
).copy()

# Classify agreement and disagreement
comparison_df["comparison"] = comparison_df.apply(
    lambda row: (
        "AGREE"
        if row["ml_prediction"] == row["np3_alert"]
        else "ML_ONLY"
        if row["ml_prediction"] == 1
        else "NP3_ONLY"
    ),
    axis=1
)

# Count each category
comparison_counts = comparison_df["comparison"].value_counts()

print("NP3 vs ML COMPARISON")
print("--------------------")
print(comparison_counts)

# Percentages
comparison_percentages = (
    comparison_counts / len(comparison_df) * 100
).round(2)

print("\nPERCENTAGES")
print("-----------")
print(comparison_percentages)

print("\nTotal comparison rows:", len(comparison_df))

NP3 vs ML COMPARISON
--------------------
comparison
AGREE       204231
NP3_ONLY     16137
ML_ONLY       9630
Name: count, dtype: int64

PERCENTAGES
-----------
comparison
AGREE       88.80
NP3_ONLY     7.02
ML_ONLY      4.19
Name: count, dtype: float64

Total comparison rows: 229998


### 209. Write three observations about where the model adds value and where it does not.

1. The ML model adds value by identifying additional risk cases that the NP3 rule does not flag. The ML-only cases were 10.83% of the test data.

2. The ML model and NP3 rule agree on most cases, with 87.47% agreement. This shows that the model provides a similar signal to the existing rule in many cases.

3. The ML model is not a replacement for the rule. NP3 still identified 1.70% of cases that the ML model missed, so both approaches have limitations and should be used as investigation signals rather than proof of congestion.


In [23]:
# 217. Save the trained ML3 model for FastAPI model serving.

import joblib
from pathlib import Path

model_path = Path(r"D:\NOPIS\ml\v2_logistic_regression.joblib")

joblib.dump(model, model_path)

print("Model saved successfully.")
print("Model path:", model_path)
print("Model type:", type(model).__name__)

Model saved successfully.
Model path: D:\NOPIS\ml\v2_logistic_regression.joblib
Model type: LogisticRegression


In [24]:
# Verify that the saved model can be loaded successfully.

loaded_model = joblib.load(model_path)

print("Model loaded successfully.")
print("Model type:", type(loaded_model).__name__)

Model loaded successfully.
Model type: LogisticRegression


In [26]:
# Save ML3 model metadata

import json
from pathlib import Path

metadata = {
    "model_version": "ml3_v2.0",
    "model_type": "LogisticRegression",
    "features": [
        "avg_activity",
        "activity_growth",
        "active_hours",
        "peak_ratio",
        "variability",
        "internet_share_avg"
    ],
    "target": "next_hour_high_activity",
    "target_threshold": 1893.40913,
    "threshold_percentile": 95
}

metadata_path = Path(r"D:\NOPIS\ml\ml3_v2_model_metadata.json")

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata saved successfully.")
print("Metadata path:", metadata_path)

print("\nMetadata:")
print(json.dumps(metadata, indent=4))

Metadata saved successfully.
Metadata path: D:\NOPIS\ml\ml3_v2_model_metadata.json

Metadata:
{
    "model_version": "ml3_v2.0",
    "model_type": "LogisticRegression",
    "features": [
        "avg_activity",
        "activity_growth",
        "active_hours",
        "peak_ratio",
        "variability",
        "internet_share_avg"
    ],
    "target": "next_hour_high_activity",
    "target_threshold": 1893.40913,
    "threshold_percentile": 95
}
